# score_runs

GPT-4.1 PF/SC for any DATASET/RUN produced by prompting_arm. Scorer is OmniContext's `OmniContextScore`, imported not copied, with the retry-loop `break` patch. References capped at 1024^2 and rounded to /16; generated image passed last, unresized. `with_scene=False`.

The judge is always shown `r["instruction"]` — the original row instruction — whatever the generator was fed. The task is fixed across conditions; only the model input varies.

Overall = per-item sqrt(PF*SC), then mean. Output: `scores/{DATASET}__{RUN}.jsonl`. Resumable; failures stay retryable. Scores whichever rows have images, so a partial run (e.g. a 20-row determinism rerun) needs no special handling.

Section 6 gives the paired per-row delta against BASE_RUN and splits sigma_d against the judge-only floor. Delta ICC across rows sharing a target was measured at ~0 for the prompting deltas used here, so target clustering is negligible for this test; it is not negligible for every contrast — adapter deltas cluster at roughly 0.02 to 0.11 and are checked with cluster-robust sensitivity. Raw scores cluster (ICC 0.185 on test, 0.096 on screen) and absolute means need cluster-robust intervals.

302 rows = 604 calls

## 1. Repo, scorer patch

In [ ]:
import os, subprocess, sys
REPO = "/content/OmniGen2"
if not os.path.isdir(REPO):
    subprocess.run(["git","clone","https://github.com/VectorSpaceLab/OmniGen2.git",REPO], check=True)
subprocess.run([sys.executable,"-m","pip","install","-q","regex"], check=True)   # json_util imports `regex`
sys.path.insert(0, REPO)
os.chdir(REPO)

# without the break, evaluate() re-runs a successful parse to max_tries: 6 calls per row instead of 2
p = "/content/OmniGen2/omnicontext/omnicontext_score.py"
lines = open(p).read().split("\n")
if not any("added: stop after successful parse" in l for l in lines):
    out = []
    for ln in lines:
        out.append(ln)
        if 'raise Exception("rate_limit_exceeded")' in ln:
            out.append("                break   # added: stop after successful parse")
    open(p, "w").write("\n".join(out))
    print("patched")
else:
    print("already patched")

patched


## 2. Drive, run config

In [ ]:
from google.colab import drive, userdata
import os, json
drive.mount("/content/drive")
os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")

DRIVE    = "/content/drive/MyDrive/Colab Notebooks/My Master's Thesis/Data"
DATASET  = "screen"        # screen_small | screen | test
RUN      = "n_contrastive2"   # fixed for this notebook
BASE_RUN = "baseline"      # comparator for section 6

GEN    = f"{DRIVE}/gen/{DATASET}/{RUN}"
SCORES = f"{DRIVE}/scores/{DATASET}__{RUN}.jsonl"
os.makedirs(os.path.dirname(SCORES), exist_ok=True)

have = sorted(int(f[:-4]) for f in os.listdir(GEN) if f.endswith(".png") and f[:-4].isdigit())
assert have, f"no generated images in {GEN}"
print(f"images : {GEN} -> {len(have)} (rows {have[0]}..{have[-1]})")
print(f"scores : {SCORES}")
man = f"{GEN}/manifest.json"
if os.path.exists(man):
    m = json.load(open(man))
    print(f"manifest: {m.get('gpu')} | torch {m.get('torch')} | written {m.get('written')}")

print(f"config  : {str(m.get('instruction_row0'))[:100]}")
print(f"negative: {'shipped default' if m.get('negative_prompt','').startswith('(((deformed)))') else 'CUSTOM'}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
images : /content/drive/MyDrive/Colab Notebooks/My Master's Thesis/Data/gen/screen/n_contrastive2 -> 302 (rows 0..301)
scores : /content/drive/MyDrive/Colab Notebooks/My Master's Thesis/Data/scores/screen__n_contrastive2.jsonl
manifest: NVIDIA A100-SXM4-80GB | torch 2.6.0+cu124 | written 2026-07-18 19:56:22
config  : A focused businesswoman sits poised in a modern office chair, her attention directed towards the sle
negative: shipped default


## 3. Reference images

In [ ]:
IMGS, EXPECTED = "/content/imgs", 37063
TAR = f"{DRIVE}/imgs_subset.tar"

def n_files(d): return sum(len(f) for _, _, f in os.walk(d)) if os.path.isdir(d) else 0

if n_files(IMGS) < EXPECTED:
    os.makedirs(IMGS, exist_ok=True)
    r = subprocess.run(f'cp "{TAR}" /content/imgs_subset.tar && tar -xf /content/imgs_subset.tar -C {IMGS}',
                       shell=True, capture_output=True, text=True)
    if r.returncode: raise RuntimeError(f"rc={r.returncode}\n{r.stderr[:800]}")
    print("restored", n_files(IMGS), "/", EXPECTED)
    assert n_files(IMGS) >= EXPECTED, "restore incomplete"
else:
    print("images complete:", n_files(IMGS))

for s in ["train", "val", "test", "screen", "screen_small"]:
    src = f"{DRIVE}/{s}.jsonl"
    if os.path.exists(src) and not os.path.exists(f"/content/{s}.jsonl"):
        subprocess.run(["cp", src, f"/content/{s}.jsonl"])

rows = [json.loads(l) for l in open(f"/content/{DATASET}.jsonl", encoding="utf-8")]
assert max(have) < len(rows), f"image index {max(have)} outside {DATASET} ({len(rows)} rows)"
missing = [p for i in have for p in rows[i]["input_images"] if not os.path.exists(os.path.join(IMGS, p))]
assert not missing, f"{len(missing)} reference images missing, first: {missing[:3]}"
print(f"{DATASET}: {len(rows)} rows | scoring {len(have)}")

restored 37063 / 37063
screen: 302 rows | scoring 302


## 4. Sanity — 3 rows

In [ ]:
import math
from PIL import Image
from omnicontext.omnicontext_score import OmniContextScore

scorer = OmniContextScore("https://api.openai.com/v1/chat/completions", userdata.get("OPENAI_API_KEY"))

def resize_ref(im):   # OmniContext default: cap at 1024^2, round to /16 — references only
    w, h = im.size
    cap = 1024*1024
    if w*h > cap:
        r = math.sqrt(cap/(w*h)); w, h = int(w*r), int(h*r)
    return im.resize((max(16, w//16*16), max(16, h//16*16)))

for i in have[:3]:
    r = rows[i]
    refs = [resize_ref(Image.open(os.path.join(IMGS, p)).convert("RGB")) for p in r["input_images"]]
    gen  = Image.open(f"{GEN}/{i:05d}.png").convert("RGB")
    res  = scorer.evaluate(refs + [gen], r["instruction"], with_scene=False)
    print(f"row {i}: PF={res['PF_scores']['score']}  SC={res['SC_scores']['score']}")
    print(f"   PF: {str(res['PF_scores']['reasoning'])[:140]}")
    print(f"   SC: {str(res['SC_scores']['reasoning'])[:140]}\n")

row 0: PF=8  SC=5
   PF: The image mostly fulfills the instruction: A businesswoman sits in a modern office chair, blazer is visible, she's focused on a laptop with 
   SC: The notepad in the final image is similar in general form to the one in the reference (spiral-bound, lined, white pages), but key details do

row 1: PF=8  SC=9
   PF: The image successfully implements most elements of the instruction: a businesswoman in a tailored blazer is seated in a modern office chair,
   SC: The laptop in the generated image closely matches the sleek laptop from the reference image in terms of color, shape, and overall appearance

row 2: PF=10  SC=9
   PF: The businesswoman is depicted sitting in a modern office chair, dressed in a tailored blazer, and exudes professionalism. She is focused on 
   SC: The businesswoman in the final image closely matches the subject from the original image. Facial features such as the eyes, eyebrows, lips, 



## 5. Full pass (resumable)

In [ ]:
import time
from concurrent.futures import ThreadPoolExecutor, as_completed

def num(x):           # fix_json can repair a bare score into [8]
    if isinstance(x, list) and x: x = x[0]
    return x if isinstance(x, (int, float)) else None

done = set()
if os.path.exists(SCORES):
    for l in open(SCORES, encoding="utf-8"):
        try:
            d = json.loads(l)
            if "error" not in d: done.add(d["i"])   # failures stay retryable
        except: pass

def score_row(i):
    r  = rows[i]
    gp = f"{GEN}/{i:05d}.png"
    if not os.path.exists(gp): return {"i": i, "error": "missing_generated"}
    refs = [resize_ref(Image.open(os.path.join(IMGS, p)).convert("RGB")) for p in r["input_images"]]
    gen  = Image.open(gp).convert("RGB")            # official: generated image not resized
    time.sleep(0.2)
    res = scorer.evaluate(refs + [gen], r["instruction"], with_scene=False)   # original instruction, never the augmented one
    pf, sc = num(res["PF_scores"]["score"]), num(res["SC_scores"]["score"])
    if pf is None or sc is None: return {"i": i, "error": "unparsable_score"}
    ups = ["_up" in os.path.basename(p) for p in r["input_images"]]
    return {"i": i, "uid": r["output_image"] + "|" + "|".join(sorted(r["input_images"])), "n_refs": len(r["input_images"]),
            "ref_kind": "all_up" if all(ups) else ("no_up" if not any(ups) else "mixed"),
            "PF": pf, "SC": sc,
            "PF_reason": res["PF_scores"]["reasoning"], "SC_reason": res["SC_scores"]["reasoning"]}

todo = [i for i in have if i not in done]
print(f"to score: {len(todo)} (done: {len(done)})")
with open(SCORES, "a", encoding="utf-8") as f, ThreadPoolExecutor(max_workers=8) as ex:
    futs = [ex.submit(score_row, i) for i in todo]
    for n, fut in enumerate(as_completed(futs), 1):
        f.write(json.dumps(fut.result(), ensure_ascii=False) + "\n"); f.flush()
        if n % 25 == 0: print(f"  {n}/{len(todo)}")
print("scoring done ->", SCORES)

to score: 302 (done: 0)
  25/302
  50/302
  75/302
  100/302
  125/302
  150/302
  175/302
  200/302
  225/302
  250/302
  275/302
  300/302
scoring done -> /content/drive/MyDrive/Colab Notebooks/My Master's Thesis/Data/scores/screen__n_contrastive2.jsonl


## 6. Aggregate, paired delta vs BASE_RUN

In [ ]:
import math, statistics as st
from collections import defaultdict

SIGMA_ITEM = 0.362      # judge re-scoring SD on an identical image; RMS over sd_reps (90 rows x5, ddof=1)

def load(run):
    d, errs = {}, defaultdict(int)
    for l in open(f"{DRIVE}/scores/{DATASET}__{run}.jsonl", encoding="utf-8"):
        r = json.loads(l)
        if "error" in r: errs[r["error"]] += 1; continue
        d[r["i"]] = r
    return d, errs

ov = lambda d: math.sqrt(max(d["PF"], 0) * max(d["SC"], 0))
cur, errs = load(RUN)
print(f"{DATASET}/{RUN}: {len(cur)} scored" + (f" | errors {dict(errs)}" if errs else ""))
print(f"  PF {st.mean([d['PF'] for d in cur.values()]):.3f}  "
      f"SC {st.mean([d['SC'] for d in cur.values()]):.3f}  "
      f"Overall {st.mean([ov(d) for d in cur.values()]):.3f}")
for key in ["n_refs", "ref_kind"]:
    g = defaultdict(list)
    for d in cur.values(): g[d[key]].append(ov(d))
    print("  by " + key + ": " + " | ".join(f"{k}: {st.mean(v):.3f} (n={len(v)})" for k, v in sorted(g.items(), key=str)))

if RUN != BASE_RUN:
    from scipy.stats import ttest_rel
    base, _ = load(BASE_RUN)
    I = sorted(set(cur) & set(base))
    # a row index means nothing on its own; the uid is what identifies the data row
    bad = [i for i in I if cur[i]["uid"] != base[i]["uid"]]
    assert not bad, f"uid mismatch on rows {bad[:5]} — runs are not aligned"
    print(f"\npaired delta vs {BASE_RUN} (n={len(I)}, uid-verified)")
    for lab, fn in [("PF", lambda d: d["PF"]), ("SC", lambda d: d["SC"]), ("Overall", ov)]:
        d = [fn(cur[i]) - fn(base[i]) for i in I]
        t, p = ttest_rel([fn(cur[i]) for i in I], [fn(base[i]) for i in I])
        print(f"  {lab:8s} mean {st.mean(d):+.3f}  SD {st.stdev(d):.3f}  "
              f"SE {st.stdev(d)/math.sqrt(len(d)):.3f}  t {t:+.2f}  p {p:.3f}")

    sd    = st.stdev([ov(cur[i]) - ov(base[i]) for i in I])
    floor = math.sqrt(2) * SIGMA_ITEM
    excess = math.sqrt(max(sd*sd - floor*floor, 0))
    print(f"\n  sigma_d (Overall)   = {sd:.3f}")
    print(f"  judge-only floor    = {floor:.3f}   (sqrt(2) * {SIGMA_ITEM})")
    print(f"  excess over floor   = {excess:.3f}")
    print(f"  -> implied per-image sigma if this pair differs only by generation noise: {excess/math.sqrt(2):.3f}")
    print("     (valid only when RUN re-runs BASE_RUN's config; otherwise excess = generation + prompt effect)")
    for delta in [0.1, 0.2, 0.3]:
        print(f"  rows for 80% power at true delta {delta}: {math.ceil(7.85*sd*sd/(delta*delta))}")
else:
    print("\nbaseline run: no comparator")

screen/n_contrastive2: 302 scored
  PF 7.960  SC 8.119  Overall 7.937
  by n_refs: 1: 7.910 (n=152) | 2: 7.980 (n=120) | 3: 7.904 (n=30)
  by ref_kind: all_up: 7.203 (n=119) | mixed: 8.356 (n=73) | no_up: 8.453 (n=110)

paired delta vs baseline (n=302, uid-verified)
  PF       mean -0.056  SD 0.874  SE 0.050  t -1.12  p 0.264
  SC       mean +0.040  SD 1.262  SE 0.073  t +0.55  p 0.585
  Overall  mean -0.018  SD 1.016  SE 0.058  t -0.31  p 0.759

  sigma_d (Overall)   = 1.016
  judge-only floor    = 0.512   (sqrt(2) * 0.362)
  excess over floor   = 0.877
  -> implied per-image sigma if this pair differs only by generation noise: 0.620
     (valid only when RUN re-runs BASE_RUN's config; otherwise excess = generation + prompt effect)
  rows for 80% power at true delta 0.1: 810
  rows for 80% power at true delta 0.2: 203
  rows for 80% power at true delta 0.3: 90
